# Sistemas de Big Data — Análisis de las preguntas de negocio REE 2019–2025

## Objetivo

Este notebook convierte las **7 preguntas de negocio seleccionadas** en análisis ejecutables sobre el CSV de REE.

El flujo será:

**CSV → Pandas → cálculos → tablas → gráficos → CSV de resultados → InfluxDB → Grafana**

Las preguntas principales son:

1. **P1 — Renovable vs. nuclear:** ¿Cómo ha cambiado la composición de la generación sin emisiones?
2. **P2 — Renovable vs. emisiones:** ¿Cuando aumenta la participación renovable, también baja la generación con emisiones?
3. **P3 — Tecnologías:** ¿Qué tecnologías muestran un crecimiento estructural y cuáles una evolución más irregular?
4. **P4 — Potencia vs. generación:** ¿El crecimiento de la generación eólica y solar es proporcional al aumento de potencia instalada?
5. **P5 — Territorio:** ¿Qué comunidades presentan mayor desequilibrio entre generación y demanda?
6. **P6 — Hidráulica vs. emisiones:** Cuando la hidráulica disminuye, ¿qué cambios se observan en la generación con emisiones?
7. **P7 — Intercambios:** ¿Cómo evoluciona el saldo eléctrico con Francia, Portugal, Marruecos y Andorra?

Además, **P0 — Calidad de datos** se ejecuta antes de sacar conclusiones.

> Las conclusiones reales se obtienen después de ejecutar los cálculos. No se anticipan resultados.

## Nota sobre los datos territoriales

El documento de preguntas indica incidencias en **Navarra, Comunidad de Madrid y Región de Murcia**. Por prudencia, este notebook utiliza para los análisis territoriales las **14 comunidades restantes** y añade Ceuta y Melilla cuando corresponde.

Esto no significa que los datos problemáticos se borren del CSV original. Simplemente se excluyen de los cálculos territoriales principales hasta poder validarlos con la fuente de REE.

# 0. Preparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Archivo original
archivo = Path("dataset_ree_limpio_transicion_energetica_2019_2025_csv.csv")

# Carpeta donde guardaremos los resultados para InfluxDB/Grafana
carpeta_salida = Path("salidas_ree")
carpeta_salida.mkdir(exist_ok=True)

df = pd.read_csv(archivo)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Registros:", len(df))
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

# P0. Control de calidad

Antes de utilizar los resultados comprobamos:

- registros marcados para revisión,
- duplicados,
- unidades,
- granularidad,
- territorios,
- y posibles problemas territoriales conocidos.

In [ ]:
print("=== QUALITY FLAG ===")
print(df["quality_flag"].value_counts(dropna=False))

print("\n=== DUPLICADOS ===")
print(df.duplicated().sum())

print("\n=== UNIDADES ===")
print(df["value_unit"].value_counts(dropna=False))

print("\n=== GRANULARIDAD ===")
print(df["time_granularity"].value_counts(dropna=False))

print("\n=== TERRITORIOS ===")
print(df[["geography_type", "geography_name"]].drop_duplicates().sort_values(
    ["geography_type", "geography_name"]
))

In [ ]:
# Territorios utilizados en los análisis territoriales.
# Excluimos los tres territorios señalados en el documento como problemáticos.

territorios_problematicos = [
    "Navarra",
    "Comunidad de Madrid",
    "Región de Murcia"
]

ccaa = sorted(df.loc[df["geography_type"] == "ccaa", "geography_name"].dropna().unique())

ccaa_validas = [
    x for x in ccaa
    if x not in territorios_problematicos
]

territorios_validos = ccaa_validas + ["Ceuta", "Melilla"]

print("CCAA válidas:", len(ccaa_validas))
print(ccaa_validas)
print("\nTerritorios utilizados:", len(territorios_validos))
print(territorios_validos)

# P1. Renovable frente a nuclear dentro de la generación sin emisiones

### Pregunta

**¿Cómo ha cambiado la composición de la generación sin emisiones entre 2019 y 2025 y qué parte de ese cambio procede de las energías renovables frente a la nuclear?**

Para evitar duplicidades utilizamos:

- `estructura_renovables` → `Generación renovable`
- `estructura_generacion` → `Nuclear`

Después sumamos por fecha y calculamos el total sin emisiones como:

**renovable + nuclear**

In [ ]:
# Generación renovable mensual
p1_ren = df[
    (df["endpoint"] == "estructura_renovables") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["indicator_name"] == "Generación renovable")
].groupby("date", as_index=False)["value"].sum()

p1_ren = p1_ren.rename(columns={"value": "renovable_MWh"})

# Generación nuclear mensual
p1_nuc = df[
    (df["endpoint"] == "estructura_generacion") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["technology_name"] == "Nuclear")
].groupby("date", as_index=False)["value"].sum()

p1_nuc = p1_nuc.rename(columns={"value": "nuclear_MWh"})

p1 = p1_ren.merge(p1_nuc, on="date", how="outer").fillna(0)

p1["sin_emisiones_MWh"] = p1["renovable_MWh"] + p1["nuclear_MWh"]
p1["renovable_pct_sin_emisiones"] = (
    p1["renovable_MWh"] / p1["sin_emisiones_MWh"] * 100
)
p1["nuclear_pct_sin_emisiones"] = (
    p1["nuclear_MWh"] / p1["sin_emisiones_MWh"] * 100
)

p1["year"] = p1["date"].dt.year
p1 = p1.sort_values("date")

p1.head()

In [ ]:
# Resumen anual de P1

p1_anual = (
    p1.groupby("year", as_index=False)
      .agg(
          renovable_MWh=("renovable_MWh", "sum"),
          nuclear_MWh=("nuclear_MWh", "sum")
      )
)

p1_anual["sin_emisiones_MWh"] = (
    p1_anual["renovable_MWh"] + p1_anual["nuclear_MWh"]
)

p1_anual["renovable_pct"] = (
    p1_anual["renovable_MWh"] / p1_anual["sin_emisiones_MWh"] * 100
)

p1_anual["nuclear_pct"] = (
    p1_anual["nuclear_MWh"] / p1_anual["sin_emisiones_MWh"] * 100
)

p1_anual

In [ ]:
# Comparación 2019 frente a 2025

p1_comparacion = p1_anual[p1_anual["year"].isin([2019, 2025])].copy()

p1_comparacion["cambio_2019_2025_MWh"] = np.nan
p1_comparacion

In [ ]:
# Calculamos el cambio entre 2019 y 2025

fila_2019 = p1_anual[p1_anual["year"] == 2019].iloc[0]
fila_2025 = p1_anual[p1_anual["year"] == 2025].iloc[0]

print("Cambio renovable MWh:", fila_2025["renovable_MWh"] - fila_2019["renovable_MWh"])
print("Cambio nuclear MWh:", fila_2025["nuclear_MWh"] - fila_2019["nuclear_MWh"])
print("Cambio sin emisiones MWh:", fila_2025["sin_emisiones_MWh"] - fila_2019["sin_emisiones_MWh"])
print("Cambio peso renovable (puntos porcentuales):",
      fila_2025["renovable_pct"] - fila_2019["renovable_pct"])

In [ ]:
# Gráfico P1: evolución de la generación sin emisiones

plt.figure(figsize=(10, 5))
plt.plot(p1_anual["year"], p1_anual["renovable_MWh"], marker="o", label="Renovable")
plt.plot(p1_anual["year"], p1_anual["nuclear_MWh"], marker="o", label="Nuclear")
plt.plot(p1_anual["year"], p1_anual["sin_emisiones_MWh"], marker="o", label="Sin emisiones")
plt.title("P1 — Evolución de la generación sin emisiones")
plt.xlabel("Año")
plt.ylabel("MWh")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Gráfico P1: composición porcentual

plt.figure(figsize=(10, 5))
plt.plot(p1_anual["year"], p1_anual["renovable_pct"], marker="o", label="Renovable")
plt.plot(p1_anual["year"], p1_anual["nuclear_pct"], marker="o", label="Nuclear")
plt.title("P1 — Composición de la generación sin emisiones")
plt.xlabel("Año")
plt.ylabel("%")
plt.ylim(0, 100)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Guardamos el resultado de P1 para usarlo posteriormente en InfluxDB/Grafana

p1_influx = p1[
    [
        "date",
        "renovable_MWh",
        "nuclear_MWh",
        "sin_emisiones_MWh",
        "renovable_pct_sin_emisiones",
        "nuclear_pct_sin_emisiones"
    ]
].copy()

p1_influx.to_csv(carpeta_salida / "P1_renovable_nuclear.csv", index=False)
p1_influx.head()

# P2. Renovable frente a generación con emisiones

### Pregunta

**Cuando aumenta la participación renovable, ¿se observa al mismo tiempo una reducción de la generación con emisiones o ambas pueden crecer en determinados periodos?**

Comparamos por mes:

- generación renovable,
- generación con emisiones,
- generación sin emisiones,
- participación renovable,
- variaciones.

In [ ]:
p2_mix = df[
    (df["endpoint"] == "evolucion_generacion_emisiones") &
    (df["geography_name"].isin(territorios_validos))
].pivot_table(
    index="date",
    columns="indicator_name",
    values="value",
    aggfunc="sum"
).reset_index()

p2_ren = df[
    (df["endpoint"] == "evolucion_renovable_no_renovable") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["indicator_name"] == "Renovable")
].groupby("date", as_index=False)["value"].sum()

p2_ren = p2_ren.rename(columns={"value": "renovable_MWh"})

p2 = p2_mix.merge(p2_ren, on="date", how="left")

p2["generacion_total_MWh"] = (
    p2["Con emisiones de CO2 eq."] +
    p2["Sin emisiones de CO2 eq."]
)

p2["renovable_pct_total"] = (
    p2["renovable_MWh"] / p2["generacion_total_MWh"] * 100
)

p2["variacion_renovable_pct"] = p2["renovable_MWh"].pct_change() * 100
p2["variacion_emisiones_pct"] = (
    p2["Con emisiones de CO2 eq."].pct_change() * 100
)

p2["year"] = p2["date"].dt.year
p2 = p2.sort_values("date")

p2.head()

In [ ]:
# Evolución anual para P2

p2_anual = (
    p2.groupby("year", as_index=False)
      .agg(
          renovable_MWh=("renovable_MWh", "sum"),
          emisiones_MWh=("Con emisiones de CO2 eq.", "sum"),
          sin_emisiones_MWh=("Sin emisiones de CO2 eq.", "sum"),
          generacion_total_MWh=("generacion_total_MWh", "sum")
      )
)

p2_anual["renovable_pct_total"] = (
    p2_anual["renovable_MWh"] /
    p2_anual["generacion_total_MWh"] * 100
)

p2_anual

In [ ]:
# Gráfico P2: renovable frente a generación con emisiones

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(p2_anual["year"], p2_anual["renovable_MWh"], marker="o", label="Renovable")
ax1.plot(p2_anual["year"], p2_anual["emisiones_MWh"], marker="o", label="Con emisiones")
ax1.set_xlabel("Año")
ax1.set_ylabel("MWh")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(
    p2_anual["year"],
    p2_anual["renovable_pct_total"],
    marker="s",
    linestyle="--",
    label="% renovable"
)
ax2.set_ylabel("% renovable")

fig.suptitle("P2 — Evolución de renovables y generación con emisiones")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.show()

In [ ]:
# Periodos en los que renovable y emisiones suben o bajan en el mismo sentido

p2_periodos = p2[
    ["date", "renovable_MWh", "Con emisiones de CO2 eq.",
     "variacion_renovable_pct", "variacion_emisiones_pct"]
].dropna().copy()

p2_periodos["mismo_sentido"] = np.where(
    np.sign(p2_periodos["variacion_renovable_pct"]) ==
    np.sign(p2_periodos["variacion_emisiones_pct"]),
    "Sí",
    "No"
)

p2_periodos.sort_values("date").tail(20)

In [ ]:
p2_influx = p2[
    [
        "date",
        "renovable_MWh",
        "Con emisiones de CO2 eq.",
        "Sin emisiones de CO2 eq.",
        "generacion_total_MWh",
        "renovable_pct_total"
    ]
].copy()

p2_influx = p2_influx.rename(columns={
    "Con emisiones de CO2 eq.": "emisiones_MWh",
    "Sin emisiones de CO2 eq.": "sin_emisiones_MWh"
})

p2_influx.to_csv(carpeta_salida / "P2_renovable_emisiones.csv", index=False)

# P3. Tecnologías con crecimiento estructural vs. evolución irregular

### Pregunta

**¿Qué tecnologías muestran un crecimiento estructural y cuáles presentan una evolución más irregular entre 2019 y 2025?**

Calcularemos:

- generación anual,
- crecimiento acumulado 2019–2025,
- variación interanual,
- desviación de las variaciones interanuales.

Una tecnología que crece mucho pero con oscilaciones fuertes no debe interpretarse igual que una tecnología que mantiene una tendencia más estable.

In [ ]:
tecnologias = df[
    (df["endpoint"] == "estructura_generacion") &
    (df["geography_name"].isin(territorios_validos))
].groupby(
    ["date", "technology_name"],
    as_index=False
)["value"].sum()

tecnologias["year"] = tecnologias["date"].dt.year

p3_anual = (
    tecnologias.groupby(["year", "technology_name"], as_index=False)["value"]
    .sum()
    .rename(columns={"value": "generacion_MWh"})
)

p3_anual["variacion_interanual_pct"] = (
    p3_anual.groupby("technology_name")["generacion_MWh"]
    .pct_change() * 100
)

p3_anual.head(20)

In [ ]:
# Resumen de crecimiento y variabilidad

resumen_p3 = (
    p3_anual.groupby("technology_name")
    .agg(
        generacion_2019_MWh=("generacion_MWh", lambda x: x.iloc[0] if len(x) else np.nan),
        generacion_2025_MWh=("generacion_MWh", lambda x: x.iloc[-1] if len(x) else np.nan),
        variabilidad_interanual_pct=("variacion_interanual_pct", "std")
    )
    .reset_index()
)

resumen_p3["crecimiento_acumulado_pct"] = (
    (resumen_p3["generacion_2025_MWh"] /
     resumen_p3["generacion_2019_MWh"] - 1) * 100
)

resumen_p3 = resumen_p3.replace([np.inf, -np.inf], np.nan)
resumen_p3.sort_values("crecimiento_acumulado_pct", ascending=False)

In [ ]:
# Elegimos tecnologías importantes para visualizar.
# Si alguna no aparece, simplemente no tendrá línea.

tecnologias_clave = [
    "Eólica",
    "Solar fotovoltaica",
    "Hidráulica",
    "Ciclo combinado",
    "Carbón",
    "Cogeneración"
]

p3_grafico = p3_anual[
    p3_anual["technology_name"].isin(tecnologias_clave)
]

plt.figure(figsize=(11, 6))

for tecnologia in tecnologias_clave:
    datos = p3_grafico[p3_grafico["technology_name"] == tecnologia]
    if not datos.empty:
        plt.plot(
            datos["year"],
            datos["generacion_MWh"],
            marker="o",
            label=tecnologia
        )

plt.title("P3 — Evolución de las principales tecnologías")
plt.xlabel("Año")
plt.ylabel("Generación (MWh)")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
resumen_p3.to_csv(carpeta_salida / "P3_tendencias_tecnologias.csv", index=False)

# P4. Potencia instalada frente a generación

### Pregunta

**La potencia eólica y solar ha aumentado: ¿el crecimiento de la generación ha sido proporcional al aumento de la potencia instalada o cambia el rendimiento aparente de la capacidad añadida?**

Construimos:

**generación anual / potencia instalada anual**

Este indicador se interpreta como **generación por MW instalado**, no como eficiencia técnica exacta.

In [ ]:
# Generación mensual de eólica y solar fotovoltaica

p4_gen = df[
    (df["endpoint"] == "estructura_generacion") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["technology_name"].isin(["Eólica", "Solar fotovoltaica"]))
].groupby(
    ["date", "technology_name"],
    as_index=False
)["value"].sum()

p4_gen = p4_gen.rename(columns={"value": "generacion_MWh"})

# Potencia instalada mensual

p4_pot = df[
    (df["endpoint"] == "potencia_instalada") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["technology_name"].isin(["Eólica", "Solar fotovoltaica"]))
].groupby(
    ["date", "technology_name"],
    as_index=False
)["value"].sum()

p4_pot = p4_pot.rename(columns={"value": "potencia_MW"})

p4 = p4_gen.merge(
    p4_pot,
    on=["date", "technology_name"],
    how="inner"
)

p4["generacion_por_MW"] = (
    p4["generacion_MWh"] / p4["potencia_MW"]
)

p4["year"] = p4["date"].dt.year

p4_anual = (
    p4.groupby(["year", "technology_name"], as_index=False)
      .agg(
          generacion_MWh=("generacion_MWh", "sum"),
          potencia_MW=("potencia_MW", "mean")
      )
)

p4_anual["generacion_por_MW"] = (
    p4_anual["generacion_MWh"] / p4_anual["potencia_MW"]
)

p4_anual

In [ ]:
# Gráfico P4

for tecnologia in ["Eólica", "Solar fotovoltaica"]:
    datos = p4_anual[p4_anual["technology_name"] == tecnologia]

    plt.figure(figsize=(10, 5))
    plt.plot(
        datos["year"],
        datos["generacion_por_MW"],
        marker="o"
    )
    plt.title(f"P4 — Generación por MW instalado: {tecnologia}")
    plt.xlabel("Año")
    plt.ylabel("MWh por MW instalado")
    plt.grid(True)
    plt.show()

In [ ]:
# Comparación del crecimiento de potencia y generación

p4_crecimiento = p4_anual.copy()

for tecnologia in ["Eólica", "Solar fotovoltaica"]:
    datos = p4_crecimiento[
        p4_crecimiento["technology_name"] == tecnologia
    ].copy()

    if not datos.empty:
        base = datos.iloc[0]

        datos["crecimiento_generacion_pct"] = (
            datos["generacion_MWh"] / base["generacion_MWh"] - 1
        ) * 100

        datos["crecimiento_potencia_pct"] = (
            datos["potencia_MW"] / base["potencia_MW"] - 1
        ) * 100

        plt.figure(figsize=(10, 5))
        plt.plot(
            datos["year"],
            datos["crecimiento_generacion_pct"],
            marker="o",
            label="Generación"
        )
        plt.plot(
            datos["year"],
            datos["crecimiento_potencia_pct"],
            marker="o",
            label="Potencia instalada"
        )
        plt.title(f"P4 — Crecimiento relativo: {tecnologia}")
        plt.xlabel("Año")
        plt.ylabel("Crecimiento desde el primer año (%)")
        plt.grid(True)
        plt.legend()
        plt.show()

In [ ]:
p4_anual.to_csv(carpeta_salida / "P4_potencia_vs_generacion.csv", index=False)

# P5. Diferencias territoriales entre generación y demanda

### Pregunta

**¿Qué comunidades presentan un mayor desequilibrio entre la electricidad que generan y la que demandan, y cómo ha evolucionado ese desequilibrio desde 2019?**

Calculamos:

- generación total,
- demanda,
- saldo `generación - demanda`,
- ratio `generación / demanda`.

No interpretamos el saldo como una exportación física exacta.

In [ ]:
# Generación total por territorio y mes

p5_gen = df[
    (df["endpoint"] == "estructura_generacion") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["technology_name"] == "Generación total")
].groupby(
    ["date", "geography_name"],
    as_index=False
)["value"].sum()

p5_gen = p5_gen.rename(columns={"value": "generacion_MWh"})

# Demanda por territorio y mes

p5_dem = df[
    (df["endpoint"] == "demanda_evolucion") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["indicator_name"] == "Demanda")
].groupby(
    ["date", "geography_name"],
    as_index=False
)["value"].sum()

p5_dem = p5_dem.rename(columns={"value": "demanda_MWh"})

p5 = p5_gen.merge(
    p5_dem,
    on=["date", "geography_name"],
    how="inner"
)

p5["saldo_generacion_demanda_MWh"] = (
    p5["generacion_MWh"] - p5["demanda_MWh"]
)

p5["ratio_generacion_demanda"] = (
    p5["generacion_MWh"] / p5["demanda_MWh"]
)

p5["year"] = p5["date"].dt.year

p5_anual = (
    p5.groupby(["year", "geography_name"], as_index=False)
      .agg(
          generacion_MWh=("generacion_MWh", "sum"),
          demanda_MWh=("demanda_MWh", "sum")
      )
)

p5_anual["saldo_MWh"] = (
    p5_anual["generacion_MWh"] - p5_anual["demanda_MWh"]
)

p5_anual["ratio_generacion_demanda"] = (
    p5_anual["generacion_MWh"] /
    p5_anual["demanda_MWh"]
)

p5_anual.head(20)

In [ ]:
# Ranking territorial en 2025

p5_2025 = (
    p5_anual[p5_anual["year"] == 2025]
    .sort_values("saldo_MWh", ascending=False)
)

p5_2025

In [ ]:
# Gráfico P5: territorios con mayor excedente/déficit en 2025

datos = p5_2025.sort_values("saldo_MWh")

plt.figure(figsize=(10, 7))
plt.barh(datos["geography_name"], datos["saldo_MWh"])
plt.axvline(0, linewidth=1)
plt.title("P5 — Saldo generación - demanda por territorio (2025)")
plt.xlabel("MWh")
plt.ylabel("Territorio")
plt.grid(axis="x")
plt.show()

In [ ]:
# Evolución de las 5 comunidades con mayor saldo absoluto en 2025

top5 = (
    p5_2025.assign(abs_saldo=lambda x: x["saldo_MWh"].abs())
    .nlargest(5, "abs_saldo")["geography_name"]
    .tolist()
)

for territorio in top5:
    datos = p5_anual[p5_anual["geography_name"] == territorio]

    plt.figure(figsize=(9, 4))
    plt.plot(datos["year"], datos["saldo_MWh"], marker="o")
    plt.axhline(0, linewidth=1)
    plt.title(f"P5 — Evolución del saldo generación-demanda: {territorio}")
    plt.xlabel("Año")
    plt.ylabel("MWh")
    plt.grid(True)
    plt.show()

In [ ]:
p5_anual.to_csv(carpeta_salida / "P5_generacion_demanda_territorial.csv", index=False)

# P6. Hidráulica frente a generación con emisiones

### Pregunta

**En los periodos en los que la generación hidráulica disminuye de forma significativa, ¿qué cambios se observan en la generación de tecnologías con emisiones?**

Aquí buscamos una **asociación observable**, no una causalidad.

Calcularemos las variaciones mensuales y compararemos:

- cambio de hidráulica,
- cambio de generación con emisiones,
- comportamiento conjunto.

In [ ]:
# Hidráulica mensual

p6_hid = df[
    (df["endpoint"] == "estructura_generacion") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["technology_name"] == "Hidráulica")
].groupby("date", as_index=False)["value"].sum()

p6_hid = p6_hid.rename(columns={"value": "hidraulica_MWh"})

# Generación con emisiones mensual

p6_em = df[
    (df["endpoint"] == "evolucion_generacion_emisiones") &
    (df["geography_name"].isin(territorios_validos)) &
    (df["indicator_name"] == "Con emisiones de CO2 eq.")
].groupby("date", as_index=False)["value"].sum()

p6_em = p6_em.rename(columns={"value": "emisiones_MWh"})

p6 = p6_hid.merge(p6_em, on="date", how="inner").sort_values("date")

p6["variacion_hidraulica_MWh"] = p6["hidraulica_MWh"].diff()
p6["variacion_emisiones_MWh"] = p6["emisiones_MWh"].diff()

p6["variacion_hidraulica_pct"] = p6["hidraulica_MWh"].pct_change() * 100
p6["variacion_emisiones_pct"] = p6["emisiones_MWh"].pct_change() * 100

p6.head()

In [ ]:
# Seleccionamos los periodos con una caída especialmente grande de hidráulica.
# Aquí utilizamos el percentil 25 para definir "caída relevante" de forma reproducible.

umbral_hidraulica = p6["variacion_hidraulica_pct"].quantile(0.25)

p6_caidas = p6[
    p6["variacion_hidraulica_pct"] <= umbral_hidraulica
].copy()

print("Umbral de caída hidráulica (%):", umbral_hidraulica)
p6_caidas[
    [
        "date",
        "variacion_hidraulica_pct",
        "variacion_emisiones_pct"
    ]
].sort_values("date")

In [ ]:
# Gráfico P6: relación entre variaciones

plt.figure(figsize=(9, 6))
plt.scatter(
    p6["variacion_hidraulica_pct"],
    p6["variacion_emisiones_pct"]
)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("P6 — Variación hidráulica vs. variación de generación con emisiones")
plt.xlabel("Variación hidráulica (%)")
plt.ylabel("Variación generación con emisiones (%)")
plt.grid(True)
plt.show()

In [ ]:
# Correlación descriptiva

correlacion_p6 = p6[
    ["variacion_hidraulica_pct", "variacion_emisiones_pct"]
].corr()

correlacion_p6

In [ ]:
p6.to_csv(carpeta_salida / "P6_hidraulica_vs_emisiones.csv", index=False)

# P7. Intercambios eléctricos internacionales

### Pregunta

**¿Cómo ha evolucionado el saldo de los intercambios eléctricos con Francia, Portugal, Marruecos y Andorra y en qué momentos cambia España de una posición importadora a exportadora?**

Aquí sí utilizamos las series de **intercambios físicos diarios** del sistema eléctrico.

Para cada país:

**saldo = exportación - importación**

Por tanto:

- saldo > 0 → posición exportadora,
- saldo < 0 → posición importadora.

In [ ]:
paises = {
    "Francia": "intercambios_francia_fisicos",
    "Portugal": "intercambios_portugal_fisicos",
    "Marruecos": "intercambios_marruecos_fisicos",
    "Andorra": "intercambios_andorra_fisicos"
}

p7_lista = []

for pais, endpoint in paises.items():

    datos = df[
        (df["endpoint"] == endpoint) &
        (df["geography_name"] == "Sistema eléctrico") &
        (df["indicator_name"] == "saldo")
    ][["date", "value"]].copy()

    datos["pais"] = pais
    datos = datos.rename(columns={"value": "saldo_MWh"})

    p7_lista.append(datos)

p7 = pd.concat(p7_lista, ignore_index=True)
p7 = p7.sort_values(["pais", "date"])

p7["year"] = p7["date"].dt.year
p7["month"] = p7["date"].dt.month

p7.head()

In [ ]:
# Saldo anual por país

p7_anual = (
    p7.groupby(["year", "pais"], as_index=False)["saldo_MWh"]
    .sum()
)

p7_anual

In [ ]:
# Detectamos el primer cambio de signo del saldo anual

cambios_p7 = []

for pais in p7_anual["pais"].unique():

    datos = p7_anual[p7_anual["pais"] == pais].sort_values("year").copy()

    datos["signo"] = np.where(
        datos["saldo_MWh"] > 0,
        "Exportador",
        np.where(datos["saldo_MWh"] < 0, "Importador", "Equilibrado")
    )

    datos["signo_anterior"] = datos["signo"].shift()

    cambios = datos[
        datos["signo"] != datos["signo_anterior"]
    ].copy()

    cambios["pais"] = pais
    cambios_p7.append(cambios)

p7_cambios_signo = pd.concat(cambios_p7, ignore_index=True)

p7_cambios_signo

In [ ]:
# Gráfico P7

plt.figure(figsize=(11, 6))

for pais in paises:
    datos = p7_anual[p7_anual["pais"] == pais]

    plt.plot(
        datos["year"],
        datos["saldo_MWh"],
        marker="o",
        label=pais
    )

plt.axhline(0, linewidth=1)
plt.title("P7 — Saldo anual de intercambios físicos")
plt.xlabel("Año")
plt.ylabel("Saldo (MWh)")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Exportamos P7 para InfluxDB/Grafana

p7.to_csv(carpeta_salida / "P7_intercambios_internacionales_diarios.csv", index=False)
p7_anual.to_csv(carpeta_salida / "P7_intercambios_internacionales_anuales.csv", index=False)

# 14. Resumen de resultados que llevaremos a InfluxDB

Para Grafana no necesitamos guardar todo el CSV original otra vez.

La idea es almacenar las **métricas ya calculadas** que queremos visualizar.

Cada resultado tendrá como mínimo:

- fecha,
- métrica,
- valor,
- y, cuando corresponda, tecnología, territorio o país.

Los CSV anteriores quedan guardados en `salidas_ree/`.

In [ ]:
# Creamos una estructura común para resultados de series temporales.
# Esto facilita posteriormente la carga en InfluxDB.

salidas = []

# P1
for _, r in p1.iterrows():
    for metrica in [
        "renovable_MWh",
        "nuclear_MWh",
        "sin_emisiones_MWh",
        "renovable_pct_sin_emisiones",
        "nuclear_pct_sin_emisiones"
    ]:
        salidas.append({
            "time": r["date"],
            "measurement": "ree_p1",
            "metric": metrica,
            "value": r[metrica]
        })

# P2
for _, r in p2.iterrows():
    for metrica in [
        "renovable_MWh",
        "emisiones_MWh",
        "sin_emisiones_MWh",
        "generacion_total_MWh",
        "renovable_pct_total"
    ]:
        if metrica in r:
            salidas.append({
                "time": r["date"],
                "measurement": "ree_p2",
                "metric": metrica,
                "value": r[metrica]
            })

influx_base = pd.DataFrame(salidas)

influx_base.to_csv(
    carpeta_salida / "resultados_base_influx.csv",
    index=False
)

influx_base.head()

# 15. Preparación opcional para InfluxDB

Esta parte **no necesita ejecutarse todavía** si no tenemos configurados:

- URL de InfluxDB,
- token,
- organización,
- bucket.

Cuando estén disponibles, podremos escribir los resultados directamente desde Python.

La estructura recomendada será:

**measurement** → pregunta (`ree_p1`, `ree_p2`, etc.)

**tags** → tecnología, territorio, país, etc.

**fields** → valor numérico de la métrica.

**time** → fecha.

No ponemos todos los datos originales en InfluxDB: primero reducimos el dataset a las métricas que realmente queremos visualizar.

In [ ]:
# OPCIONAL — instalar si no tienes la librería:
# %pip install influxdb-client

# from influxdb_client import InfluxDBClient, Point, WritePrecision
# from influxdb_client.client.write_api import SYNCHRONOUS

# INFLUX_URL = "http://localhost:8086"
# INFLUX_TOKEN = "TU_TOKEN"
# INFLUX_ORG = "TU_ORG"
# INFLUX_BUCKET = "ree"

# client = InfluxDBClient(
#     url=INFLUX_URL,
#     token=INFLUX_TOKEN,
#     org=INFLUX_ORG
# )

# write_api = client.write_api(write_options=SYNCHRONOUS)

# for _, fila in influx_base.iterrows():
#     point = (
#         Point(fila["measurement"])
#         .tag("metric", fila["metric"])
#         .field("value", float(fila["value"]))
#         .time(fila["time"], WritePrecision.NS)
#     )
#     write_api.write(
#         bucket=INFLUX_BUCKET,
#         org=INFLUX_ORG,
#         record=point
#     )

# client.close()

# 16. Qué podremos visualizar después en Grafana

Con las salidas de este notebook podemos construir dashboards como:

### Dashboard 1 — Transición energética
- Renovable vs. nuclear.
- % renovable dentro de generación sin emisiones.
- Evolución de generación con emisiones.

### Dashboard 2 — Tecnologías
- Eólica.
- Solar fotovoltaica.
- Hidráulica.
- Carbón.
- Ciclo combinado.
- Cogeneración.

### Dashboard 3 — Potencia y generación
- MW instalados.
- MWh generados.
- MWh por MW instalado.

### Dashboard 4 — Territorio
- Generación.
- Demanda.
- Saldo generación-demanda.
- Ranking de comunidades.

### Dashboard 5 — Intercambios internacionales
- Francia.
- Portugal.
- Marruecos.
- Andorra.
- Saldo importador/exportador.

---

## Resultado del notebook

Al ejecutar todas las celdas se generará la carpeta:

`salidas_ree/`

con los resultados preparados para el siguiente paso del proyecto.

**La interpretación final debe hacerse mirando los resultados reales obtenidos por las tablas y gráficos.**